In [52]:
import pandas as pd
import matplotlib.pyplot as plt

/home/edi/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [53]:
df = pd.read_json("companies.jsonl", lines=True)

In [5]:
print(df.head())

         website operational_name  year_founded  \
0   rompetrol.ro        Rompetrol        1979.0   
1    unilever.ro         Unilever           NaN   
2  rompetrol.com        Rompetrol        1974.0   
3        cbre.ro     CBRE Romania        2008.0   
4  timacagro.com       TIMAC AGRO           NaN   

                                             address  employee_count  \
0  {'country_code': 'ro', 'latitude': 44.4775537,...             NaN   
1  {'country_code': 'ro', 'latitude': 44.4761769,...             NaN   
2  {'country_code': 'ro', 'latitude': 44.47775356...          7000.0   
3  {'country_code': 'ro', 'latitude': 44.4540414,...            92.0   
4  {'country_code': 'ro', 'latitude': 44.4792186,...            65.0   

        revenue                                      primary_naics  \
0  1.349824e+10  {'code': '324110', 'label': 'Petroleum Refiner...   
1  1.280207e+10  {'code': '424690', 'label': 'Other Chemical an...   
2  1.266496e+10  {'code': '324110', 'label': 'Petr

In [6]:
revenues = df['revenue']
websites = df['website'] 
print(revenues.head(10))
print(websites.head(10))

0    1.349824e+10
1    1.280207e+10
2    1.266496e+10
3    9.568739e+09
4    8.907855e+09
5    6.542287e+09
6    5.290918e+09
7    4.801924e+09
8    4.495347e+09
9    3.492537e+09
Name: revenue, dtype: float64
0                   rompetrol.ro
1                    unilever.ro
2                  rompetrol.com
3                        cbre.ro
4                  timacagro.com
5                         omv.ro
6                         cfr.ro
7    mercedes-benz-trucks-mbb.ro
8                       dacia.ro
9                      fildas.ro
Name: website, dtype: object


In [7]:
print(df.iloc[44])

website                                     mangalmoneytransfer.co.uk
operational_name                                Mangal Money Transfer
year_founded                                                   2013.0
address             {'country_code': 'gb', 'country_name': 'United...
employee_count                                                    4.0
revenue                                                      385783.0
primary_naics       {'code': '522320', 'label': 'Financial Transac...
description         V Send Ltd, DBA Mangal Money Transfer, is a UK...
business_model      [Service Provider, Business-to-Business, Busin...
target_markets                                   [Financial Services]
core_offerings      [Partner Network Money Transfer Services, Invo...
is_public                                                       False
secondary_naics     {'code': '522390', 'label': 'Other Activities ...
Name: 44, dtype: object


In [31]:
raw_prompts = """
    Logistic companies in Romania
    Public software companies with more than 1,000 employees.
    Food and beverage manufacturers in France
    Companies that could supply packaging materials for a direct-to-consumer cosmetics brand
    Construction companies in the United States with revenue over $50 million
    Pharmaceutical companies in Switzerland
    B2B SaaS companies providing HR solutions in Europe
    Clean energy startups founded after 2018 with fewer than 200 employees
    Fast-growing fintech companies competing with traditional banks in Europe.
    E-commerce companies using Shopify or similar platforms
    Renewable energy equipment manufacturers in Scandinavia
    Companies that manufacture or supply critical components for electric vehicle battery production """

list_raw_prompts = raw_prompts.split('\n')[1:]
prompts = []
for prompt in list_raw_prompts:
    prompts.append(prompt.strip())
print(prompts)

['Logistic companies in Romania', 'Public software companies with more than 1,000 employees.', 'Food and beverage manufacturers in France', 'Companies that could supply packaging materials for a direct-to-consumer cosmetics brand', 'Construction companies in the United States with revenue over $50 million', 'Pharmaceutical companies in Switzerland', 'B2B SaaS companies providing HR solutions in Europe', 'Clean energy startups founded after 2018 with fewer than 200 employees', 'Fast-growing fintech companies competing with traditional banks in Europe.', 'E-commerce companies using Shopify or similar platforms', 'Renewable energy equipment manufacturers in Scandinavia', 'Companies that manufacture or supply critical components for electric vehicle battery production']


In [33]:
import ollama
import json
import re

def extract_json_from_query(query):
    system_prompt = """
    You are a highly precise data extraction API. Your ONLY job is to extract search constraints from a user query and format them into a strict JSON object.
    
    CRITICAL RULES FOR COMPARATORS:
    - "over", "more than", "exceeding", "greater than" -> map to `min_revenue` or `min_employees`.
    - "under", "less than", "fewer than" -> map to `max_revenue` or `max_employees`.
    - "after", "since" -> map to `year_founded_after`.
    - "prior to", "before" -> map to `year_founded_before`.
    
    INSTRUCTIONS:
    1. ONLY extract a constraint if explicitly stated. Do not guess.
    2. If a constraint is not mentioned, its value MUST be `null` or `[]`.
    3. You MUST start your response with a `_reasoning` key, briefly explaining how you map the numbers based on the comparators.
    4. Start your response immediately with `{`. No markdown formatting outside the JSON.

    Use EXACTLY this JSON structure:
    {
      "_reasoning": "Brief explanation of how you mapped the comparators to the keys.",
      "hard_filters": {
        "location": null, // CRITICAL: MUST be a single string or null. NEVER a list [].
        "min_revenue": null, // CRITICAL: Integer only.
        "max_revenue": null,
        "min_employees": null,
        "max_employees": null,
        "year_founded_after": null,
        "year_founded_before": null,
        "is_public": null
      },
      "soft_filters": {
        "industry_or_vertical": [],
        "business_model": [],
        "core_offerings": []
      }
    }

    Example 1:
    Query: "Startups with less than 50 employees founded prior to 2015"
    Output:
    {
      "_reasoning": "'less than 50 employees' means max_employees is 50. 'prior to 2015' means year_founded_before is 2015.",
      "hard_filters": {
        "location": null, "min_revenue": null, "max_revenue": null, "min_employees": null, "max_employees": 50, "year_founded_after": null, "year_founded_before": 2015, "is_public": null
      },
      "soft_filters": { "industry_or_vertical": ["startups"], "business_model": [], "core_offerings": [] }
    }
    
    Example 2:
    Query: "Public software companies with more than 1,000 employees."
    Output:
    {
      "_reasoning": "'Public' means is_public is true. 'more than 1,000 employees' means min_employees is 1000.",
      "hard_filters": {
        "location": null, "min_revenue": null, "max_revenue": null, "min_employees": 1000, "max_employees": null, "year_founded_after": null, "year_founded_before": null, "is_public": true
      },
      "soft_filters": { "industry_or_vertical": ["software"], "business_model": [], "core_offerings": [] }
    }
    """
    # Critical rule for extracting only what is clear, not interpreting
    # Should use soft filtering instead of hard filtering for this reason: 
    #        -> If value hard to interpret, use hard filter
    #        -> If value ambiguous, use soft filter later in cosine similarity
    
    response = ollama.chat(
        model = "llama3",
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f'Query: "{query}"\nOutput:'}
        ],
        options = {
            "temperature": 0.0
        },
        format="json"
    )

    # Extract the raw output and clean from llm output
    raw_output = response["message"]["content"]
    cleaned_output = raw_output.strip()
    if cleaned_output.startswith("```json"):
        cleaned_output = cleaned_output[7:]
    if cleaned_output.startswith("```"):
        cleaned_output = cleaned_output[3:]
    if cleaned_output.endswith("```"):
        cleaned_output = cleaned_output[:-3]

    # Try to parse the output as JSON
    try:
        parsed_json = json.loads(cleaned_output.strip())
        return parsed_json
    except json.JSONDecodeError:
        print(f"Failed to parse JSON. Raw output was: {raw_output}")
        return None

In [35]:
# for query in prompts:
#     print(f"\nCurrent querry: {query}")
query = "B2B SaaS companies providing HR solutions in Europe"
json_query = extract_json_from_query(query)
print(json.dumps(json_query, indent = 2))

{
  "_reasoning": "'B2B' and 'SaaS' are not constraints, so they're ignored. 'providing HR solutions' is a soft filter for core_offerings. 'in Europe' means location should be Europe.",
  "hard_filters": {
    "location": "Europe",
    "min_revenue": null,
    "max_revenue": null,
    "min_employees": null,
    "max_employees": null,
    "year_founded_after": null,
    "year_founded_before": null,
    "is_public": null
  },
  "soft_filters": {
    "industry_or_vertical": [
      "B2B",
      "SaaS"
    ],
    "business_model": [],
    "core_offerings": [
      "HR solutions"
    ]
  }
}


In [48]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# The sentences to encode
sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)

comparator = "There is perfect temperature outside"
comparator_embedding = model.encode([comparator])

similarities = cosine_similarity(comparator_embedding, embeddings)[0]
print(similarities)


Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 10662.52it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[0.5544592  0.59621716 0.09537424]]


In [122]:
import ast
class Searcher:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def prepare_company_text(self, company):
        attributes = []

        # Get description
        if pd.notna(company.get("description")):
            description = company["description"]
            attributes.append(str(description))

        # Get primary naics
        if pd.notna(company.get("primary_naics")):
            primary_naics_data = company["primary_naics"]
            primary_naics_dict = {}
            if isinstance(primary_naics_data, str):
                primary_naics_dict = ast.literal_eval(primary_naics_data)    
            elif isinstance(primary_naics_data, dict):
                primary_naics_dict = primary_naics_data

            attributes.append(f"Industry: {primary_naics_dict["label"]}")

        # Get core offerings
        if len(company.get("core_offerings")) > 0:
            core_offerings = company["core_offerings"]
            core_offerings = ", ".join(core_offerings)

            attributes.append(f"Offering: {core_offerings}")

        if len(company.get("target_markets")) > 0:
            target_markets = company["target_markets"]
            target_markets = ", ".join(target_markets)

            attributes.append(target_markets)
        
        return " | ".join(attributes)
        
    def rank_companies(self, companies, query, top_k):
        if companies.empty:
            return companies

        company_attributes = companies.apply(self.prepare_company_text, axis = 1).tolist()

        companies_embeddings = self.model.encode(company_attributes, show_progress_bar = False)
        query_embedding = self.model.encode([query])

        similarities = cosine_similarity(query_embedding, companies_embeddings)[0]
        
        ranked_companies = companies.copy()
        ranked_companies["score"] = similarities

        ranked_companies = ranked_companies.sort_values(by="score", ascending = False)

        return ranked_companies.head(top_k)

In [124]:
searcher = Searcher()

top_companies = searcher.rank_companies(df, "Fuel Product Distribution companies", 10)
print(top_companies)


Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 11908.08it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


                  website        operational_name  year_founded  \
16              oscars.ro                   OSCAR        2001.0   
5                  omv.ro                     OMV           NaN   
2           rompetrol.com               Rompetrol        1974.0   
0            rompetrol.ro               Rompetrol        1979.0   
244         aromas.com.au  Aromas Coffee Roasters        1982.0   
249                  None                 Vetipak        1997.0   
18              valtec.nl                  Valtec           NaN   
316              styri.eu                   Styri        2025.0   
247  centralenvasados.com    Central de Envasados        1988.0   
14            transgaz.ro                Transgaz        2000.0   

                                               address  employee_count  \
16   {'country_code': 'ro', 'latitude': 44.4471231,...           488.0   
5    {'country_code': 'ro', 'latitude': 44.490331, ...             NaN   
2    {'country_code': 'ro', 'latitude': 

In [84]:
import ast
data = ast.literal_eval("{'code': '237310', 'label': 'Highway, Street, and Bridge Construction'}")
print(data["label"])

Highway, Street, and Bridge Construction
